In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!lscpu | grep -E "Model name|CPU\(s\)|cache" || true
!pip install -q mpi4py numba "numba-cuda[cu12]" numpy
!python3 -c "from numba import cuda; assert cuda.is_available(), 'GPU not attached'; cuda.detect()"

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-c81fa74e-1da1-db21-d6e2-56a8ebbb00bf, 7.5, 15360 MiB
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
L1d cache:                               32 KiB (1 instance)
L1i cache:                               32 KiB (1 instance)
L2 cache:                                1 MiB (1 instance)
L3 cache:                                38.5 MiB (1 instance)
NUMA node0 CPU(s):                       0,1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.14.1+cu130 requires cuda-bindings<14,>=13.0.3; platform_system == "Linux" and python_version < "3.15", but you have cuda-bindings 12.9.10 which is incompatible.
torch 2.14.1+cu130 requires cuda-toolkit[cublas,cudart,cufft,cufile,cupti,curand,cusolver,cuspa

In [2]:
GITHUB_USERNAME = "Serikyoyo"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
"Set GITHUB_USERNAME to your real GitHub handle before running."
)
print(f"Verified solo student handle: {GITHUB_USERNAME}")

Verified solo student handle: Serikyoyo


In [7]:
%%writefile hybrid_mpi_cuda_cache.py
import os
import re
import sys
import json
import math
import time
import hashlib
import platform
import warnings
import subprocess
import numpy as np
from mpi4py import MPI
from llvmlite import ir
from numba import cuda, njit, float64, uint64, types
from numba.core import cgutils
from numba.core.errors import NumbaPerformanceWarning
from numba.extending import intrinsic

warnings.filterwarnings("ignore", category=NumbaPerformanceWarning)

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2  # 18x18 shared tile
GLOBAL_NX = 512
GLOBAL_NY = 512
LOCAL_NX = GLOBAL_NX // 2  # 256
LOCAL_NY = GLOBAL_NY // 2  # 256
PADDED_NX = LOCAL_NX + 2   # 258
PADDED_NY = LOCAL_NY + 2   # 258
STENCIL_STEPS = 300
SHM_ITERS = 40_000_000
LAMBDA_DAMP = 0.15

TAG_FROM_DOWN = 101
TAG_FROM_UP = 102
TAG_FROM_RIGHT = 103
TAG_FROM_LEFT = 104


def get_gpu_uuid() -> str:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def get_gpu_name() -> str:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def detect_cpu_line_size() -> int:
    path = "/sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size"
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return int(f.read().strip())
    return 64


# ==============================================================================
# TASK 1: Hybrid MPI Cartesian Grid + Node-Local MPI-3 Shared-Memory Window
# ==============================================================================
def init_hybrid_context(github_user: str) -> dict:
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Requires exactly 4 MPI ranks (2x2 grid), got {size}.")

    if not cuda.is_available():
        raise RuntimeError(
            "No CUDA GPU detected. In Colab, switch via Runtime -> Change runtime type -> T4 GPU."
        )

    cuda.select_device(0)
    gpu_uuid = comm.bcast(get_gpu_uuid() if rank == 0 else None, root=0)
    gpu_name = comm.bcast(get_gpu_name() if rank == 0 else None, root=0)
    cpu_sig = f"{platform.machine()}::{os.cpu_count()}"
    line_bytes = detect_cpu_line_size()

    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}::{cpu_sig}".encode("utf-8")
    ).hexdigest()
    instance_seed = int(digest[:16], 16)

    cart_comm = comm.Create_cart(dims=[2, 2], periods=[False, False], reorder=False)
    cart_rank = cart_comm.Get_rank()
    coords = cart_comm.Get_coords(cart_rank)  # (coord_y, coord_x)
    up, down = cart_comm.Shift(direction=0, disp=1)
    left, right = cart_comm.Shift(direction=1, disp=1)

    node_comm = comm.Split_type(MPI.COMM_TYPE_SHARED)
    node_rank = node_comm.Get_rank()
    node_size = node_comm.Get_size()
    if node_size != 4:
        raise RuntimeError(f"Expected all 4 ranks on 1 node, got node_size={node_size}.")

    alloc_words = 72 if node_rank == 0 else 1
    win = MPI.Win.Allocate_shared(alloc_words * 8, 8, info=MPI.INFO_NULL, comm=node_comm)
    buf, _ = win.Shared_query(0)
    raw_arr = np.frombuffer(buf, dtype=np.uint64)
    base_addr = int(raw_arr.__array_interface__["data"][0])
    align_offset_bytes = (64 - (base_addr % 64)) % 64
    align_offset_words = align_offset_bytes // 8
    shm_view = raw_arr[align_offset_words : align_offset_words + 64]
    aligned_addr = int(shm_view.__array_interface__["data"][0])
    assert aligned_addr % 64 == 0, f"Shared window not 64-byte aligned: {aligned_addr}"

    if node_rank == 0:
        shm_view[:] = 0
    node_comm.Barrier()

    return {
        "comm": comm,
        "rank": rank,
        "size": size,
        "cart_comm": cart_comm,
        "cart_rank": cart_rank,
        "coords": coords,
        "neighbors": {"up": up, "down": down, "left": left, "right": right},
        "node_comm": node_comm,
        "node_rank": node_rank,
        "win": win,
        "shm_view": shm_view,
        "shm_aligned_addr": aligned_addr,
        "github_user": github_user.strip(),
        "gpu_name": gpu_name,
        "gpu_uuid": gpu_uuid,
        "cpu_sig": cpu_sig,
        "line_bytes": line_bytes,
        "instance_seed": instance_seed,
        "instance_seed_hex": digest[:16],
    }


# ==============================================================================
# TASK 2: Rank-Local GPU Subdomain & Halo Pack/Unpack Kernels
# ==============================================================================
@cuda.jit
def pack_halos_kernel(u, send_up, send_down, send_left, send_right):
    idx = cuda.grid(1)
    if idx < LOCAL_NX:
        i = idx + 1
        send_up[idx] = u[1, i]
        send_down[idx] = u[LOCAL_NY, i]
        send_left[idx] = u[i, 1]
        send_right[idx] = u[i, LOCAL_NX]


@cuda.jit
def unpack_halos_kernel(
    u,
    recv_up,
    recv_down,
    recv_left,
    recv_right,
    has_up,
    has_down,
    has_left,
    has_right,
):
    idx = cuda.grid(1)
    if idx < LOCAL_NX:
        i = idx + 1
        if has_up:
            u[0, i] = recv_up[idx]
        if has_down:
            u[LOCAL_NY + 1, i] = recv_down[idx]
        if has_left:
            u[i, 0] = recv_left[idx]
        if has_right:
            u[i, LOCAL_NX + 1] = recv_right[idx]


def initialize_fields(ctx: dict) -> dict:
    coord_y, coord_x = ctx["coords"]
    dx = 1.0 / (GLOBAL_NX + 1)

    seed_unit = (ctx["instance_seed"] & 0xFFFF) / 65535.0
    amp_mod = 0.95 + 0.10 * seed_unit

    u_init = np.zeros((PADDED_NY, PADDED_NX), dtype=np.float64)
    kappa = np.zeros((PADDED_NY, PADDED_NX), dtype=np.float64)
    source = np.zeros((PADDED_NY, PADDED_NX), dtype=np.float64)

    ly = np.arange(PADDED_NY, dtype=np.float64)
    lx = np.arange(PADDED_NX, dtype=np.float64)
    gy = coord_y * LOCAL_NY + (ly - 1.0)
    gx = coord_x * LOCAL_NX + (lx - 1.0)

    x = (gx + 1.0) * dx
    y = (gy + 1.0) * dx
    XX, YY = np.meshgrid(x, y, indexing="xy")

    kappa[:, :] = (
        0.50
        + 0.35 * np.sin(2.0 * math.pi * XX) * np.cos(2.0 * math.pi * YY)
        + 0.15 * np.cos(math.pi * (XX + YY))
    )

    u_full = amp_mod * (
        2.0 * np.sin(math.pi * XX) * np.sin(math.pi * YY)
        + 1.0 * np.sin(2.0 * math.pi * XX) * np.sin(2.0 * math.pi * YY)
    )
    s_full = 0.05 * np.sin(math.pi * XX) * np.sin(math.pi * YY)

    u_init[1 : LOCAL_NY + 1, 1 : LOCAL_NX + 1] = u_full[1 : LOCAL_NY + 1, 1 : LOCAL_NX + 1]
    source[1 : LOCAL_NY + 1, 1 : LOCAL_NX + 1] = s_full[1 : LOCAL_NY + 1, 1 : LOCAL_NX + 1]

    local_kappa_max = float(np.max(kappa[1 : LOCAL_NY + 1, 1 : LOCAL_NX + 1]))
    kappa_max = ctx["cart_comm"].allreduce(local_kappa_max, op=MPI.MAX)
    dt_max = (dx * dx) / (4.0 * kappa_max)
    dt = 0.80 * dt_max

    d_u = cuda.to_device(u_init)
    d_u_next = cuda.to_device(u_init)
    d_kappa = cuda.to_device(kappa)
    d_source = cuda.to_device(source)

    d_send_up = cuda.device_array((LOCAL_NX,), dtype=np.float64)
    d_send_down = cuda.device_array((LOCAL_NX,), dtype=np.float64)
    d_send_left = cuda.device_array((LOCAL_NY,), dtype=np.float64)
    d_send_right = cuda.device_array((LOCAL_NY,), dtype=np.float64)

    d_recv_up = cuda.device_array((LOCAL_NX,), dtype=np.float64)
    d_recv_down = cuda.device_array((LOCAL_NX,), dtype=np.float64)
    d_recv_left = cuda.device_array((LOCAL_NY,), dtype=np.float64)
    d_recv_right = cuda.device_array((LOCAL_NY,), dtype=np.float64)

    h_send_up = cuda.pinned_array((LOCAL_NX,), dtype=np.float64)
    h_send_down = cuda.pinned_array((LOCAL_NX,), dtype=np.float64)
    h_send_left = cuda.pinned_array((LOCAL_NY,), dtype=np.float64)
    h_send_right = cuda.pinned_array((LOCAL_NY,), dtype=np.float64)

    h_recv_up = cuda.pinned_array((LOCAL_NX,), dtype=np.float64)
    h_recv_down = cuda.pinned_array((LOCAL_NX,), dtype=np.float64)
    h_recv_left = cuda.pinned_array((LOCAL_NY,), dtype=np.float64)
    h_recv_right = cuda.pinned_array((LOCAL_NY,), dtype=np.float64)

    return {
        "u_init": u_init,
        "dx": dx,
        "kappa_max": kappa_max,
        "dt_max": dt_max,
        "dt": dt,
        "d_u": d_u,
        "d_u_next": d_u_next,
        "d_kappa": d_kappa,
        "d_source": d_source,
        "d_send_up": d_send_up,
        "d_send_down": d_send_down,
        "d_send_left": d_send_left,
        "d_send_right": d_send_right,
        "d_recv_up": d_recv_up,
        "d_recv_down": d_recv_down,
        "d_recv_left": d_recv_left,
        "d_recv_right": d_recv_right,
        "h_send_up": h_send_up,
        "h_send_down": h_send_down,
        "h_send_left": h_send_left,
        "h_send_right": h_send_right,
        "h_recv_up": h_recv_up,
        "h_recv_down": h_recv_down,
        "h_recv_left": h_recv_left,
        "h_recv_right": h_recv_right,
        "stream_interior": cuda.stream(),
        "stream_boundary": cuda.stream(),
    }


# ==============================================================================
# TASK 3: 2D Shared-Memory Tiled Stencil & Dual-Stream Pipeline
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(
    u,
    u_next,
    kappa,
    source,
    dt,
    inv_dx2,
    lambda_damp,
    y_start,
    y_end,
    x_start,
    x_end,
):
    s_u = cuda.shared.array((SHARED_DIM, SHARED_DIM), dtype=float64)
    s_k = cuda.shared.array((SHARED_DIM, SHARED_DIM), dtype=float64)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    tid = ty * TILE_DIM + tx

    block_y0 = y_start + cuda.blockIdx.y * TILE_DIM
    block_x0 = x_start + cuda.blockIdx.x * TILE_DIM

    for linear_idx in range(tid, SHARED_DIM * SHARED_DIM, TILE_DIM * TILE_DIM):
        sy = linear_idx // SHARED_DIM
        sx = linear_idx % SHARED_DIM
        gy = block_y0 - 1 + sy
        gx = block_x0 - 1 + sx
        if 0 <= gy < PADDED_NY and 0 <= gx < PADDED_NX:
            s_u[sy, sx] = u[gy, gx]
            s_k[sy, sx] = kappa[gy, gx]
        else:
            s_u[sy, sx] = 0.0
            s_k[sy, sx] = 0.0

    cuda.syncthreads()

    gy_out = block_y0 + ty
    gx_out = block_x0 + tx
    if gy_out < y_end and gx_out < x_end:
        sy = ty + 1
        sx = tx + 1

        u_c = s_u[sy, sx]
        u_n = s_u[sy - 1, sx]
        u_s = s_u[sy + 1, sx]
        u_w = s_u[sy, sx - 1]
        u_e = s_u[sy, sx + 1]

        k_c = s_k[sy, sx]
        k_n = 0.5 * (k_c + s_k[sy - 1, sx])
        k_s = 0.5 * (k_c + s_k[sy + 1, sx])
        k_w = 0.5 * (k_c + s_k[sy, sx - 1])
        k_e = 0.5 * (k_c + s_k[sy, sx + 1])

        div_flux = (
            k_e * (u_e - u_c)
            - k_w * (u_c - u_w)
            + k_s * (u_s - u_c)
            - k_n * (u_c - u_n)
        ) * inv_dx2

        u_next[gy_out, gx_out] = u_c + dt * (
            div_flux - lambda_damp * u_c + source[gy_out, gx_out]
        )


def _grid_for_range(y_start: int, y_end: int, x_start: int, x_end: int):
    ny = y_end - y_start
    nx = x_end - x_start
    return ((nx + TILE_DIM - 1) // TILE_DIM, (ny + TILE_DIM - 1) // TILE_DIM)


def step_serialized(ctx: dict, state: dict) -> None:
    cart_comm = ctx["cart_comm"]
    nb = ctx["neighbors"]
    up, down, left, right = nb["up"], nb["down"], nb["left"], nb["right"]

    d_u = state["d_u"]
    d_u_next = state["d_u_next"]
    inv_dx2 = 1.0 / (state["dx"] * state["dx"])

    pack_blocks = (LOCAL_NX + 127) // 128
    pack_halos_kernel[pack_blocks, 128](
        d_u, state["d_send_up"], state["d_send_down"], state["d_send_left"], state["d_send_right"]
    )
    cuda.synchronize()

    state["d_send_up"].copy_to_host(state["h_send_up"])
    state["d_send_down"].copy_to_host(state["h_send_down"])
    state["d_send_left"].copy_to_host(state["h_send_left"])
    state["d_send_right"].copy_to_host(state["h_send_right"])

    reqs = [
        cart_comm.Irecv(state["h_recv_up"], source=up, tag=TAG_FROM_UP),
        cart_comm.Irecv(state["h_recv_down"], source=down, tag=TAG_FROM_DOWN),
        cart_comm.Irecv(state["h_recv_left"], source=left, tag=TAG_FROM_LEFT),
        cart_comm.Irecv(state["h_recv_right"], source=right, tag=TAG_FROM_RIGHT),
        cart_comm.Isend(state["h_send_up"], dest=up, tag=TAG_FROM_DOWN),
        cart_comm.Isend(state["h_send_down"], dest=down, tag=TAG_FROM_UP),
        cart_comm.Isend(state["h_send_left"], dest=left, tag=TAG_FROM_RIGHT),
        cart_comm.Isend(state["h_send_right"], dest=right, tag=TAG_FROM_LEFT),
    ]
    MPI.Request.Waitall(reqs)

    state["d_recv_up"].copy_to_device(state["h_recv_up"])
    state["d_recv_down"].copy_to_device(state["h_recv_down"])
    state["d_recv_left"].copy_to_device(state["h_recv_left"])
    state["d_recv_right"].copy_to_device(state["h_recv_right"])

    unpack_halos_kernel[pack_blocks, 128](
        d_u,
        state["d_recv_up"],
        state["d_recv_down"],
        state["d_recv_left"],
        state["d_recv_right"],
        up != MPI.PROC_NULL,
        down != MPI.PROC_NULL,
        left != MPI.PROC_NULL,
        right != MPI.PROC_NULL,
    )

    grid_full = _grid_for_range(1, LOCAL_NY + 1, 1, LOCAL_NX + 1)
    tiled_stencil_kernel[grid_full, (TILE_DIM, TILE_DIM)](
        d_u,
        d_u_next,
        state["d_kappa"],
        state["d_source"],
        state["dt"],
        inv_dx2,
        LAMBDA_DAMP,
        1,
        LOCAL_NY + 1,
        1,
        LOCAL_NX + 1,
    )
    cuda.synchronize()
    state["d_u"], state["d_u_next"] = d_u_next, d_u


def step_overlapped(ctx: dict, state: dict) -> None:
    cart_comm = ctx["cart_comm"]
    nb = ctx["neighbors"]
    up, down, left, right = nb["up"], nb["down"], nb["left"], nb["right"]

    s_int = state["stream_interior"]
    s_bnd = state["stream_boundary"]
    d_u = state

Overwriting hybrid_mpi_cuda_cache.py


In [8]:
import os
import subprocess

env = os.environ.copy()
env["OMPI_MCA_btl_vader_single_copy_mechanism"] = "none"
env["PMIX_MCA_gds"] = "hash"

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "hybrid_mpi_cuda_cache.py",
    GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True, env=env)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Hybrid MPI-CUDA execution failed with code {proc.returncode}")

### Analytical & Forensic Engineering Responses

- **Question 1 (GPU Coalescing vs. CPU Cache-Line Granularity):**
  A host CPU transfers data in **64-byte MESI cache lines** (8 contiguous `float64` words), retaining lines across L1d/L2/L3 caches so strided column reads still reuse cached lines if spatial/temporal locality fits within cache capacity. In contrast, an NVIDIA T4 GPU executes SIMT warps of 32 threads whose global memory transactions are serviced in **32-byte L2 sectors** (4 sectors per **128-byte L1 cache line**). For a row-major `(258, 258)` `float64` array, consecutive column elements in `u[1:257, 1]` are separated by $258 \times 8 = 2,064\text{ bytes}$. Because every adjacent element lands in a separate 32-byte L2 sector (and separate 128-byte L1 line), reading the 256-element column directly without packing transfers $256 \times 32\text{ B} = 8,192\text{ B}$ from DRAM/L2 (and up to $256 \times 128\text{ B} = 32,768\text{ B}$ at L1 granularity) to obtain only $256 \times 8\text{ B} = 2,048\text{ B}$ of useful halo data—wasting **$6,144\text{ B}$ (75.0%) of L2 sector bandwidth** (or **$30,720\text{ B}$ / 93.75% at L1 line granularity**) per column, while strided D2H copies break contiguous PCIe DMA bursts into 256 tiny 8-byte transactions. `pack_halos_kernel` consolidates `u[1:257, 1]` and `u[1:257, 256]` into contiguous 1D `2,048`-byte buffers (`64` L2 sectors), achieving **100% coalesced PCIe DMA payload efficiency**.

- **Question 2 (Shared-Memory Tile Synchronization & PTX Audit):**
  From `tiled_stencil_kernel.inspect_asm()` targeting `[INSERT target_arch, e.g., (7, 5)]`, the exact instruction counts are:
  - `st.shared.f64`: **[INSERT st_shared_f64_count]**
  - `ld.shared.f64`: **[INSERT ld_shared_f64_count]**
  - `bar.sync`: **[INSERT bar_sync_count]**

  A $16 \times 16$ threadblock comprises $256 / 32 = 8$ warps that execute asynchronously across SIMT warp schedulers. During the cooperative load of the $18 \times 18$ shared tile (`324` elements), threads load border halo elements belonging to neighboring warps' stencil footprints. If `cuda.syncthreads()` (`bar.sync 0`) is removed, a faster warp will advance to the `ld.shared.f64` stencil phase and read uninitialized or stale shared-memory entries (`s_u[sy ± 1, sx ± 1]`, `s_k[sy ± 1, sx ± 1]`) before slower warps complete their `st.shared.f64` writes—causing a **Read-After-Write (RAW) data hazard** that corrupts interior flux calculations along warp boundaries.

- **Question 3 (Dual-Stream Compute-Communication Overlap on Shared GPU Silicon):**
  In my execution, `gpu_serialized_ms` was **[INSERT gpu_serialized_ms] ms** compared to `gpu_overlapped_ms` of **[INSERT gpu_overlapped_ms] ms** (a **[INSERT overlap_speedup]x** speedup, with cross-rank `max_diff = [INSERT max_diff] < 1e-12`). Overlapping `stream_interior` (`[2..255, 2..255]`) with `stream_boundary` hides halo pack/unpack and MPI exchange latency behind deep-interior arithmetic. However, because all 4 MPI ranks share a single physical T4 GPU via `cuda.select_device(0)` without MPS/NVLink, the 4 host processes time-slice the GPU's single hardware context and contend for the T4's shared PCIe Gen3 x16 DMA copy engine, partially serializing concurrent D2H/H2D transfers across ranks. On a multi-GPU node with 4 dedicated GPUs connected via **GPUDirect RDMA**, each rank owns an independent GPU context and DMA engine, and NICs read/write GPU BAR1 memory directly over PCIe/NVLink without staging through host pinned RAM or time-slicing compute SMs.

- **Question 4 (MPI-3 Shared-Memory MESI RFO Ping-Pong):**
  Measured wall-clock latencies across `40,000,000` volatile read-modify-write iterations in `MPI.Win.Allocate_shared`:
  - `stride = 1` (8 B spacing): **[INSERT stride_1_8B_ms] ms**
  - `stride = 8` (64 B spacing): **[INSERT stride_8_64B_ms] ms**
  - `stride = 16` (128 B spacing): **[INSERT stride_16_128B_ms] ms**
  - False-sharing slowdown ratio (`stride = 1` / `stride = 8`): **[INSERT false_sharing_slowdown_vs_64B]x** (`[INSERT false_sharing_slowdown_vs_128B]x` vs `stride = 16`).

  At `stride = 1`, `shm_view[0..3]` occupy bytes `0..31` of the **exact same 64-byte MESI cache line**. When Core 0 holds the line in **Modified (`M`)** after writing `shm_view[0]` and Core 1 attempts to write `shm_view[1]`, Core 1's L1d cache suffers a write miss and broadcasts a **Read-For-Ownership (RFO)** request on the coherency interconnect. Core 0 snoops the RFO, flushes/writes back the dirty line, and transitions its copy from **`M` $\rightarrow$ `I` (Invalid)**, while Core 1 receives ownership in **`E`/`M`** and transitions to **`M`** upon completing its store. Immediately afterward, Core 0's next iteration misses (`I`), broadcasts its own RFO, forces Core 1 from **`M` $\rightarrow$ `I`**, and reclaims **`M`**—triggering continuous cache-line ping-pong across all 4 cores.

- **Question 5 (128-Byte Spatial Prefetcher Pairing & Padding Trade-Offs):**
  Comparing `stride = 8` (**[INSERT stride_8_64B_ms] ms**) and `stride = 16` (**[INSERT stride_16_128B_ms] ms**) yields a ratio of **[INSERT prefetcher_isolation_ratio_64B_vs_128B]x**. At `stride = 8`, each rank occupies its own 64-byte cache line (`Base64 + 0, 64, 128, 192`), eliminating true 64-byte MESI false sharing; however, adjacent ranks (`Rank 0` & `Rank 1`, `Rank 2` & `Rank 3`) occupy even/odd buddy lines inside the same **128-byte aligned sector**. Hardware adjacent-cache-line spatial prefetchers automatically prefetch the paired 64-byte buddy line on cache misses, introducing mild L2/interconnect snoop traffic that `stride = 16` (128-byte isolation) eliminates. Conversely, excessive padding degrades performance in real workloads by inflating memory footprints by 8x–16x, reducing effective L1/L2 cache capacity, evicting useful data (capacity misses), and increasing TLB pressure.

- **Question 6 (CFL Stability & Distributed Energy Dissipation):**
  With $\Delta x = 1 / 513 \approx 1.949317738791 \times 10^{-3}$ and manifest $\kappa_{\max} = \text{[INSERT kappa\_max]}$, the 2D explicit diffusion stability limit is:
  $$\Delta t_{\max} = \frac{\Delta x^2}{4 \cdot \kappa_{\max}} = \frac{(1/513)^2}{4 \cdot \text{[INSERT kappa\_max]}} = \text{[INSERT dt\_max]}\text{ s}$$
  The simulation uses $\Delta t = 0.80 \, \Delta t_{\max} = \text{[INSERT dt]}\text{ s}$ (Courant number $= 0.80 < 1.0$), strictly satisfying the 2D Courant–Friedrichs–Lewy (CFL) stability condition. Across the 300-step evolution, the `cart_comm.allreduce` global $L^2$ energy dissipates monotonically due to variable-coefficient diffusion and linear damping ($\lambda_{\text{damp}} = 0.15$):
  - **Step 0:** `[INSERT energy_history["0"]]`
  - **Step 100:** `[INSERT energy_history["100"]]`
  - **Step 200:** `[INSERT energy_history["200"]]`
  - **Step 300:** `[INSERT energy_history["300"]]`